In [6]:
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
import os

api_key = os.getenv("GROQ_API_KEY")
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",  
    api_key=api_key
)

MODEL = "openai/gpt-oss-120b"

# Load dataset
df = pd.read_csv("german_credit_data.csv")

def create_prompt(row):
    return f"""
You are a financial risk analyst.

Given the following customer data:
Age: {row['Age']}
Sex: {row['Sex']}
Job: {row['Job']}
Housing: {row['Housing']}
Saving accounts: {row['Saving accounts']}
Checking account: {row['Checking account']}
Credit amount: {row['Credit amount']}
Duration: {row['Duration']}
Purpose: {row['Purpose']}

The risk label is: {row['Risk']}

Explain why this customer is classified as {row['Risk']}.
Also suggest improvements if the risk is bad.
"""

def generate_explanation(prompt):
    response = client.chat.completions.create(
        model=MODEL, 
        messages=[
            {"role": "system", "content": "You are an expert financial analyst."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

data = df.head(1)
PROMPT = create_prompt(data)
description = generate_explanation(PROMPT)



In [7]:
print(description)

**Why the model classifies this applicant as “good” risk**

| Variable | Value | What the model “sees” (typical interpretation) | Why it pushes the score toward *good* |
|----------|-------|-----------------------------------------------|---------------------------------------|
| **Age** | 67 | Older borrowers are often retired with a stable pension or other fixed income. In many credit‑risk datasets, age ≥ 60 is associated with a lower probability of default because the borrower has already demonstrated the ability to meet long‑term obligations (mortgage, utilities, etc.). | ✅ Age contributes positively. |
| **Sex** | male | Sex alone is not a strong predictor once other variables are accounted for, but in the training data males slightly out‑performed females on repayment for this particular product line. | ↗ Neutral‑to‑slight positive. |
| **Job** | 2 (coded “skilled‑blue‑collar”) | The job‑coding scheme typically ranks 1 = unskilled, 2 = skilled‑blue‑collar, 3 = white‑collar, etc. 

In [4]:
import pandas as pd
df = pd.read_csv("german_credit_data.csv")
df.head(1)

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,0,67,male,2,own,NaN,little,1169,6,radio/TV,good


In [12]:
import pandas as pd
from openai import OpenAI
import os
import gradio as gr

# API setup
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

MODEL = "openai/gpt-oss-120b"

# Load dataset
df = pd.read_csv("german_credit_data.csv")


def create_prompt(row):
    return f"""
You are explaining credit risk to a normal person with no finance knowledge.

Customer details:
Age: {row['Age']}
Job: {row['Job']}
Housing: {row['Housing']}
Savings: {row['Saving accounts']}
Checking balance: {row['Checking account']}
Loan amount: {row['Credit amount']}
Duration: {row['Duration']}
Purpose: {row['Purpose']}

Risk: {row['Risk']}

Explain in very simple and clear language why this person is considered {row['Risk']}.
- Use easy words (like talking to a beginner).
- No technical terms, no tables, no bullet points.
- Keep it short (3–5 lines).
- If the risk is bad, give 1–2 simple suggestions to improve.
"""


def analyze(index):
    try:
        row = df.iloc[int(index)]
        prompt = create_prompt(row)

        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are an expert financial analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"


# Minimal UI
gr.Interface(
    fn=analyze,
    inputs="text",
    outputs="text",
    title="Credit Risk Explainer (Enter Row Index)"
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
